# 02 · Order Segmentation

Rebuilds the four K-Means order personas on the same 95,824-order cohort used by the supervised models.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df=pd.read_csv('../data/master_orders_clean_v2.csv',low_memory=False)

## Canonical rebuild

The original seven clustering variables are retained. Seventeen rows have at least one missing clustering input; median imputation keeps the clustering cohort aligned with the 95,824-order supervised cohort.

In [ ]:
X=pd.DataFrame(index=df.index)
X['log_total_price']=np.log1p(df['total_price'])
X['log_total_freight']=np.log1p(df['total_freight'])
X['log_product_weight']=np.log1p(df['avg_product_weight_g'])
X['log_delivery_days']=np.log1p(df['delivery_time_days'].clip(lower=0))
X['delivery_vs_estimate']=df['delivery_vs_estimate_days'].clip(-40,40)
X['n_payment_installments']=df['n_payment_installments']
X['n_items']=df['n_items'].clip(upper=5)
print('rows with missing clustering input:', int(X.isna().any(axis=1).sum()))
X=X.fillna(X.median())
Xs=StandardScaler().fit_transform(X)
kmeans=KMeans(n_clusters=4,n_init=10,random_state=42).fit(Xs)
df['cluster']=kmeans.labels_
print('sample silhouette:', round(silhouette_score(Xs,kmeans.labels_,sample_size=20000,random_state=42),3))

In [ ]:
PERSONA={0:'Long Wait',1:'Multi-Item Basket',2:'Big-Ticket Planner',3:'Quick Small Buy'}
profile=(df.groupby('cluster').agg(size=('order_id','size'),bad_review_rate=('review_bad','mean'),late_rate=('delivered_late','mean'),median_total_price=('total_price','median'),median_total_freight=('total_freight','median'),median_product_weight_g=('avg_product_weight_g','median'),avg_delivery_time_days=('delivery_time_days','mean'),avg_delivery_vs_estimate_days=('delivery_vs_estimate_days','mean'),avg_installments=('n_payment_installments','mean'),avg_n_items=('n_items','mean')).reset_index())
profile.insert(1,'persona',profile['cluster'].map(PERSONA))
profile.insert(3,'share',profile['size']/len(df))
assert profile['size'].sum()==95824
profile.round(3)

The silhouette score is modest (~0.204), so these are descriptive order personas rather than sharply separated natural groups. Review outcome is used only after clustering for profiling.